# Desafio Transfer Learning

In [ ]:
!pip install tensorflow
!pip install matplotlib

In [ ]:
!python --version

Correto rodar no python versão 3.10.11

In [ ]:
import numpy as np
import tensorflow  # keras defasado lib para construção e treinamento de redes neurais
import random
import os
import zipfile

import matplotlib
import matplotlib.pyplot as plt # lib para geração de gráficos em Python.
from matplotlib.pyplot import imshow # função pra exibir imagens

from tensorflow import keras
from keras._tf_keras.keras.preprocessing import image # Módulo do Keras usado para carregar, pré-processar e aumentar imagens antes de alimentar a rede neural.
from keras._tf_keras.keras.applications.imagenet_utils import preprocess_input # Função que normaliza imagens de acordo com as exigências dos modelos pré-treinados do Keras (como VGG16)
from keras._tf_keras.keras.models import Sequential # Classe do Keras usada para criar um modelo de rede neural sequencial (onde as camadas são empilhadas uma após a outra).
from keras._tf_keras.keras.layers import Dense, Dropout, Flatten, Activation
'''
Dense: Camada totalmente conectada (fully connected).
Dropout: Técnica de regularização para evitar overfitting, desativando aleatoriamente algumas conexões durante o treinamento.
Flatten: Achata a matriz de features em um vetor unidimensional, necessário antes de passar para as camadas totalmente conectadas.
Activation: Função de ativação aplicada aos neurônios.
'''
from keras._tf_keras.keras.layers import Conv2D, MaxPooling2D
'''
Conv2D: Camada de convolução 2D para extrair características de imagens.
MaxPooling2D: Camada de pooling que reduz a dimensionalidade e melhora a eficiência computacional.
'''
from keras._tf_keras.keras.models import Model #  Classe para criar modelos personalizados e avançados no Keras, permitindo redes neurais mais complexas do que a estrutura sequencial.


Extração do arquivo e algumas validações

In [ ]:

dataset_path = r'.\data_set.zip'
extract_path = r'.\data_set'   # Caminho onde os arquivos serão extraídos

# Verifica se o arquivo ZIP existe
if os.path.exists(dataset_path):
    print(f"Tamanho do arquivo: {os.path.getsize(dataset_path)} bytes")
else:
    print("Arquivo não encontrado.")

# Verifica se o arquivo é um ZIP válido
if zipfile.is_zipfile(dataset_path):
    print("O arquivo é um ZIP válido. Extraindo os arquivos...")
    with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extração concluída!")
else:
    print("Erro: O arquivo não é um ZIP válido.")


root = extract_path

train_split, val_split = 0.7, 0.15
'''
train_split = 0.7: Define que 70% dos dados serão usados para treino.
val_split = 0.15: Define que 15% dos dados serão usados para validação.
Os 15% restantes (1 - train_split - val_split) serão usados para teste.
'''


categories = [x[0] for x in os.walk(root) if x[0]][1:]
'''
os.walk(root) percorre recursivamente todas as pastas e subpastas dentro de root.
x[0] for x in os.walk(root) if x[0] extrai apenas os nomes dos diretórios (ignorando arquivos dentro das pastas).
[1:] remove o primeiro elemento da lista, que é o próprio diretório root.
exemplo:
[
  ('data_set', [...], [...]),
  ('data_set/Finn', [...], [...]),
  ('data_set/Jake', [...], [...]),
  ('data_set/Cats', [...], [...]),
  ('data_set/Dogs', [...], [...]),
]

'''

num_classes = len(categories) # Usado para converter os rótulos em vetores one-hot e para exibir informações no resumo (define numero de classes)

print(categories)

Tamanho do arquivo: 145752064 bytes
Erro: O arquivo não é um ZIP válido.
[]


This function is useful for pre-processing the data into an image and input vector.

In [ ]:
# helper function to load image and return it and input vector
def get_image(path):
    img = image.load_img(path, target_size=(224, 224))
    '''
    Usa image.load_img() do Keras para carregar a imagem do caminho especificado (path).
    O argumento target_size=(224, 224) redimensiona a imagem para 224x224 pixels (tamanho padrão usado por redes como VGG16, ResNet, etc.).
    '''
    x = image.img_to_array(img) # Converts a PIL Image instance to a NumPy array.
    x = np.expand_dims(x, axis=0)
    '''
    Adiciona uma nova dimensão ao array para que ele fique no formato esperado por modelos pré-treinados.
    Modelos de deep learning geralmente esperam um lote (batch) de imagens como entrada, mesmo que seja uma única imagem.
    Isso transforma o array de (224, 224, 3) para (1, 224, 224, 3), onde:
    1 → Representa o número de imagens no lote (batch size de 1).
    224 x 224 x 3 → Dimensão da imagem.
    '''
    x = preprocess_input(x)
    '''
    normaliza os valores dos pixels, de acordo com os requisitos do modelo pré-treinado do Keras.
    O tipo de normalização depende do modelo:
    Para VGG16 e ResNet, subtrai a média dos pixels (centrando os valores).
    Para InceptionV3 e Xception, normaliza os valores para o intervalo [-1, 1].
    Esse passo melhora o desempenho da rede neural.
    '''

    return img, x


Função para remover imagens inválidas (OPCIONAL)

In [ ]:
# remove corrupted images
import os
from PIL import Image

def remove_corrupted_images(extract_path):
    removed_files = 0
    for root, _, files in os.walk(extract_path):
        for file in files:
            file_path = os.path.join(root, file)
            
            if os.path.splitext(file)[1].lower() not in ['.jpg', '.jpeg', '.png']:
                continue

            try:
                with Image.open(file_path) as img:
                    img.verify()
                    
            except Exception as e:
                print(f"Removendo imagem corrompida: {file_path} ({e})")
                os.remove(file_path)
                removed_files += 1

    print(f"\nTotal de imagens corrompidas removidas: {removed_files}")


remove_corrupted_images(extract_path)


Total de imagens corrompidas removidas: 0


Carrega todas as imagens da pasta


In [ ]:
data = []

for c, category in enumerate(categories):
    images = [os.path.join(dp, f) for dp, dn, filenames
              in os.walk(category) for f in filenames
              if os.path.splitext(f)[1].lower() in ['.jpg', '.png', '.jpeg']]

    print(f"Categoria: {category}, Total de imagens encontradas: {len(images)}")

    for img_path in images:
        try:
            img, x = get_image(img_path)  # Tenta abrir a imagem
            data.append({'x': np.array(x[0]), 'y': c})
        except Exception as e:
            print(f"Erro ao processar {img_path}: {e}")  # Exibe erro e continua

Randomize the data order.

In [ ]:
random.shuffle(data)

create training / validation / test split (70%, 15%, 15%)

In [ ]:
idx_val = int(train_split * len(data)) # porcentagem de dados destinada ao treinamento vezes ao número total de amostras no dataset
idx_test = int((train_split + val_split) * len(data)) # porcentagem de dados destinada a validação
train = data[:idx_val] # Dados de Treinamento (70%)
val = data[idx_val:idx_test] # Dados de Validação (15%)
test = data[idx_test:] # Dados de Teste (15%)

Separate data for labels.

In [ ]:
x_train, y_train = np.array([t["x"] for t in train]), [t["y"] for t in train]
x_val, y_val = np.array([t["x"] for t in val]), [t["y"] for t in val]
x_test, y_test = np.array([t["x"] for t in test]), [t["y"] for t in test]
print(y_test)

[]


train é uma lista de dicionários, onde cada item tem:

"x" → A imagem processada (um array NumPy de shape (224, 224, 3))
"y" → O rótulo da classe (um número inteiro representando a categoria da imagem).
Transformação:

x_train = np.array([t["x"] for t in train])
Cria um array NumPy contendo todas as imagens do conjunto de treino.
y_train = [t["y"] for t in train]
Cria uma lista com os rótulos correspondentes às imagens.

Exemplo do que train contém antes da conversão:
```
train = [
    {"x": array([...]), "y": 0},  # Primeira imagem (classe 0)
    {"x": array([...]), "y": 1},  # Segunda imagem (classe 1)
    {"x": array([...]), "y": 2},  # Terceira imagem (classe 2)
]

```

Depois da conversão:
```
x_train = np.array([...])  # Array NumPy contendo as imagens
y_train = [0, 1, 2]  # Lista de rótulos das classes
```


Pre-process the data as before by making sure it's float32 and normalized between 0 and 1.

In [ ]:
# normalização da data
x_train = x_train.astype('float32') / 255.
x_val = x_val.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.


Converção dos rótulos (labels) que estão representados como números inteiros em vetores one-hot.

In [ ]:
# convert labels to one-hot vectors
y_train = keras.utils.to_categorical(y_train, num_classes)
y_val = keras.utils.to_categorical(y_val, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)
print(y_test.shape)

O código abaixo percorre o diretório especificado para coletar os caminhos de todas as imagens com extensões válidas. Em seguida, ele seleciona 8 imagens aleatórias, as carrega redimensionadas para 224×224 pixels, as concatena horizontalmente e, por fim, exibe a imagem resultante usando Matplotli

In [ ]:
# summary
print("Carregamento concluído de %d imagens de %d categorias" % (len(data), num_classes))
print("Divisão treino/validação/teste: %d, %d, %d" % (len(x_train), len(x_val), len(x_test)))
print("Formato dos dados de treinamento: ", x_train.shape)
print("Formato dos rótulos de treinamento: ", y_train.shape)

Construnindo uma rede neural convolucional (CNN) usando a API Sequential do Keras para realizar uma tarefa de classificação de imagens.

In [ ]:
# build the network
model = Sequential()
print("Input dimensions: ",x_train.shape[1:]) # Imprime as dimensões de uma única amostra (altura, largura e canais), que serão usadas como forma de entrada na primeira camada.

# Primeiro bloco de convolução
model.add(Conv2D(32, (3, 3), input_shape=x_train.shape[1:]))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
'''
Conv2D(32, (3, 3), input_shape=x_train.shape[1:]): Adiciona uma camada convolucional com 32 filtros de tamanho 3×3. A propriedade input_shape define as dimensões esperadas de cada imagem.
Activation('relu'): Aplica a função de ativação ReLU, que introduz não-linearidades ao modelo.
MaxPooling2D(pool_size=(2, 2)): Reduz as dimensões espaciais da saída (downsampling) através de uma operação de max pooling com uma janela 2×2.
'''

model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Dropout(0.25)) # Camada de dropout com taxa de 25% para ajudar na regularização e prevenir o overfitting.

model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Conv2D(32, (3, 3)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))

model.add(Dropout(0.25))

model.add(Flatten())
model.add(Dense(256))
model.add(Activation('relu'))
'''
Flatten(): Converte as matrizes 2D resultantes das camadas convolucionais em um vetor 1D, que pode ser processado pelas camadas densas.
Dense(256): Adiciona uma camada totalmente conectada com 256 neurônios.
Activation('relu'): Aplica a função de ativação ReLU nessa camada.
'''

model.add(Dropout(0.5))

model.add(Dense(num_classes))
model.add(Activation('softmax'))
'''
Dense(num_classes): Adiciona a camada final totalmente conectada com um número de neurônios igual ao número de classes, onde cada neurônio corresponde a uma classe.
Activation('softmax'): Utiliza a função softmax para transformar as saídas em probabilidades, de forma que a soma das probabilidades seja igual a 1.
'''

model.summary()

Realizando duas etapas fundamentais no processo de treinamento de um modelo em Keras: a compilação do modelo e o treinamento (fit). 

In [ ]:
# Compilação do Modelo:
model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

# Treinamento do Modelo
history = model.fit(x_train, y_train,
                    batch_size=128,
                    epochs=10,
                    validation_data=(x_val, y_val))

In [ ]:
# Visualição das perdas e acuracias
fig = plt.figure(figsize=(16,4))
ax = fig.add_subplot(121)
ax.plot(history.history["val_loss"])
ax.set_title("validation loss")
ax.set_xlabel("epochs")

ax2 = fig.add_subplot(122)
ax2.plot(history.history["val_accuracy"])
ax2.set_title("validation accuracy")
ax2.set_xlabel("epochs")
ax2.set_ylim(0, 1)

plt.show()

In [ ]:
# Teste do modelo
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
print('Teste loss:', loss)
print('Teste accuracy:', accuracy)

# Aplicando Transfer Learning

In [ ]:

vgg = keras.applications.VGG16(weights='imagenet', include_top=True)
vgg.summary()


In [ ]:
# make a reference to VGG's input layer
inp = vgg.input

# make a new softmax layer with num_classes neurons
new_classification_layer = Dense(num_classes, activation='softmax')

# connect our new layer to the second to last layer in VGG, and make a reference to it
out = new_classification_layer(vgg.layers[-2].output)

# create a new network between inp and out
model_new = Model(inp, out)

In [ ]:
# make all layers untrainable by freezing weights (except for last layer)
for l, layer in enumerate(model_new.layers[:-1]):
    layer.trainable = False

# ensure the last layer is trainable/not frozen
for l, layer in enumerate(model_new.layers[-1:]):
    layer.trainable = True

model_new.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model_new.summary()

In [ ]:
history2 = model_new.fit(x_train, y_train, 
                         batch_size=128, 
                         epochs=10, 
                         validation_data=(x_val, y_val))



In [ ]:
fig = plt.figure(figsize=(16,4))
ax = fig.add_subplot(121)
ax.plot(history.history["val_loss"])
ax.plot(history2.history["val_loss"])
ax.set_title("validation loss")
ax.set_xlabel("epochs")

ax2 = fig.add_subplot(122)
ax2.plot(history.history["val_accuracy"])
ax2.plot(history2.history["val_accuracy"])
ax2.set_title("validation accuracy")
ax2.set_xlabel("epochs")
ax2.set_ylim(0, 1)

plt.show()

In [ ]:

loss, accuracy = model_new.evaluate(x_test, y_test, verbose=0)

print('Test loss:', loss)
print('Test accuracy:', accuracy)



In [ ]:

img, x = get_image('101_ObjectCategories/airplanes/image_0003.jpg')
probabilities = model_new.predict([x])